# Course 24 lab — CI/CD quality gates and evidence-based release governance

This notebook executes the same reusable release-control code as the tests. It distinguishes candidate claims, authenticated evidence, eligibility, deployment authorization, runtime acceptance, and recovery. Run it from `northstar-release-control`.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

sys.path.insert(0, str(Path.cwd()))
from release_app.simulation import (LATER, _approve, _safe_observations,
    prepare_candidate, run_reference)

ROOT = Path.cwd()
if not (ROOT / 'policies/release-policy.json').exists():
    raise RuntimeError('Run this notebook from northstar-release-control')

## 1. Exact identity and trusted eligibility

A candidate saying `PASS` is not evidence. The trusted path builds bytes, authenticates provenance, executes checks, resolves stored evidence, and recomputes the artifact digest. Passing produces review eligibility—not deployment authority.

In [ ]:
with TemporaryDirectory() as directory:
    plane, manifest, artifact = prepare_candidate(ROOT, directory)
    gate = plane.evaluate(manifest, artifact, LATER)
    identity_result = {
        'artifact_digest': manifest.artifact_digest,
        'manifest_digest': manifest.digest,
        'gate': gate.state,
        'deployed_before_approval': plane.environment_state()['release_id'],
    }
identity_result

## 2. Revision-bound AI evaluation

The application image can remain unchanged while the model changes. The next cell preserves old model evidence and proves that the new effective manifest is blocked.

In [ ]:
with TemporaryDirectory() as directory:
    plane, changed, artifact = prepare_candidate(
        ROOT, directory,
        manifest_changes={'model_revision': 'model-v13'},
        dependency_overrides={'ai_evaluation': {'model_revision': 'model-v12'}},
    )
    stale_model = plane.evaluate(changed, artifact, LATER)
{'state': stale_model.state, 'findings': stale_model.reason_codes}

## 3. Unknown deployment outcome

A timeout after an external effect is not failure. The fixture records `UNKNOWN_OUTCOME`, observes the target environment, and reconciles to `VERIFYING` before any retry.

In [ ]:
with TemporaryDirectory() as directory:
    plane, manifest, artifact = prepare_candidate(ROOT, directory)
    plane.evaluate(manifest, artifact, LATER)
    unknown = plane.deploy(manifest, _approve(plane, manifest), 'OP-NB-UNKNOWN', LATER, timeout_after_effect=True)
    observed = plane.environment_state()['release_id']
    reconciled = plane.reconcile_unknown('OP-NB-UNKNOWN', LATER)
    unknown_result = {
        'initial': unknown.state, 'observed_release': observed,
        'after_reconciliation': reconciled.state,
        'events': [row['state'] for row in plane.operation_events('OP-NB-UNKNOWN')],
    }
unknown_result

## 4. Runtime acceptance and containment

Infrastructure health is not behavioral acceptance. An observed human-review bypass halts the rollout, disables the feature, and preserves manual review.

In [ ]:
with TemporaryDirectory() as directory:
    plane, manifest, artifact = prepare_candidate(ROOT, directory)
    plane.evaluate(manifest, artifact, LATER)
    plane.deploy(manifest, _approve(plane, manifest), 'OP-NB-CANARY', LATER)
    halted = plane.verify_runtime('OP-NB-CANARY', [{
        'tenant_authorized': True, 'ambiguous': True,
        'human_review_required': False, 'protected_mutation': False,
    }], LATER)
    containment = {'decision': halted.state, 'findings': halted.reason_codes, 'environment': plane.environment_state()}
containment

## 5. Full evidence report

The reference report executes twelve golden and twelve adversarial fixtures. These populations prove named local invariants only; they are not production release-system accuracy or model-quality evidence.

In [ ]:
report = run_reference(ROOT)
{
    'decision': report['decision'],
    'golden': (report['golden_scenarios']['passed'], report['golden_scenarios']['total']),
    'adversarial': (report['adversarial_evaluations']['passed'], report['adversarial_evaluations']['total']),
    'operation_states': report['reference_release']['operation_states'],
    'non_authorities': report['non_authorities'],
}

## Production upgrades and exercises

Replace local HMAC provenance with an approved Sigstore/in-toto trust policy; isolate untrusted CI; add registry immutability, KMS-backed keys, durable orchestration, outbox delivery, idempotent deployment adapters, protected audit storage, statistically justified rollout windows, and privacy-aware telemetry.

Exercises: mutate the prompt, retrieval index, tool policy, schema, and dataset independently; justify targeted invalidation; add a transient failure with bounded retry; and design a complete rollback where business-state reconciliation remains open.